# Semantic Memory Extractor Evaluation with Qwen

This notebook evaluates a conservative semantic memory extraction prompt for a fitness coaching use case.

The extractor answers:

> What should a future fitness coach remember about this user?

It should not summarize the conversation. It should extract only durable, user-specific facts that are likely to remain useful for weeks or months.

The design intentionally prioritizes precision over recall. Missing a weak memory is better than storing junk.


## Notebook goals

By the end of this notebook, you will be able to:

1. Load a Hugging Face Qwen instruct model in Colab.
2. Run a conservative semantic memory extraction prompt.
3. Test the prompt on 30 realistic conversation examples.
4. Compare model outputs against expected memories.
5. Measure JSON validity, precision, recall, F1, false positives, and category accuracy.
6. Review failure cases and improve the prompt.
7. Save predictions, raw outputs, and error analysis for later comparison.


## What counts as long-term semantic memory?

A long-term semantic memory is a durable user-specific fact that could help a future coach personalize training, nutrition, motivation, or communication.

Good memories:

```text
The user prefers morning workouts.
The user dislikes burpees.
The user follows a vegetarian diet that includes eggs and dairy.
The user is lactose intolerant.
The user has a knee limitation that affects jumping exercises.
The user prefers encouraging coaching feedback.
The user has adjustable dumbbells and resistance bands at home.
```

Bad memories:

```text
The user skipped the gym today.
It was raining today.
The user ate pizza yesterday.
The user asked for a workout.
The coach suggested squats.
The user felt tired this morning.
The user's friend follows keto.
The user might start running next week.
```


## Conservative extraction policy

The extractor should store a memory only when all of these are true:

1. It is about the user.
2. It was explicitly stated by the user.
3. It is useful for future fitness, nutrition, motivation, or coaching personalization.
4. It is likely to remain true for weeks or months.
5. It is not a one-off event, temporary state, assistant suggestion, third-party fact, or irrelevant detail.

When uncertain, return no memory.

This notebook is designed to catch junk memories. In semantic memory systems, false positives are often more harmful than false negatives because they can pollute future personalization.


## Setup

Run this cell first in Google Colab.

The default model is small enough for quick testing. You can switch to a stronger Qwen model later if your GPU has enough memory.


In [ ]:
!pip install -U transformers accelerate torch pandas tqdm scikit-learn

In [ ]:
import json
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer


## Model configuration

Recommended default:

```python
Qwen/Qwen2.5-1.5B-Instruct
```

This is a practical default for Colab testing.

For better extraction quality, try:

```python
Qwen/Qwen2.5-7B-Instruct
```

The larger model will need more GPU memory.


In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

MAX_NEW_TOKENS = 768
GENERATION_TEMPERATURE = 0.0
MATCH_THRESHOLD = 0.55

OUTPUT_DIRECTORY = Path(".")
PREDICTIONS_CSV_PATH = OUTPUT_DIRECTORY / "semantic_memory_predictions.csv"
ERROR_ANALYSIS_CSV_PATH = OUTPUT_DIRECTORY / "semantic_memory_error_analysis.csv"
RAW_OUTPUTS_JSONL_PATH = OUTPUT_DIRECTORY / "semantic_memory_raw_outputs.jsonl"


## Extraction prompt

Keep the prompt in a separate cell so it can be edited and versioned easily.

The prompt contains:

1. Role definition
2. Conservative memory rules
3. Approved categories
4. Strict JSON schema
5. Durability rules
6. Realistic few-shot examples
7. Empty-output examples


In [ ]:
SYSTEM_PROMPT = """
You are a long-term semantic memory extraction assistant for a fitness coach.

Your job is to read a user conversation and extract only durable, user-specific facts that would help a future fitness coach personalize training, nutrition, motivation, or communication.

You are conservative. False memories are worse than missed memories. When uncertain, return no memory.

A memory should be extracted only if all five conditions are true:
1. It is about the user, not another person.
2. It is explicitly stated by the user.
3. It would be useful for future fitness, nutrition, motivation, or coaching personalization.
4. It is likely to remain true for weeks or months.
5. It is not a one-off event, temporary state, weather detail, assistant suggestion, third-party fact, or irrelevant chat detail.

Allowed memory categories:
- fitness_goal
- body_goal
- workout_preference
- workout_dislike
- food_preference
- dietary_restriction
- nutrition_goal
- schedule
- injury_or_limitation
- coaching_preference
- tracking_preference
- equipment
- experience_level
- lifestyle_constraint

Do not use any other category.

Strong durability signals include:
usually, normally, always, never, I prefer, I hate, I avoid, I need, I have, I am allergic, I cannot, most days, every week, for years, long-term, my doctor told me, outside Ramadan I usually.

Weak or temporary signals include:
today, yesterday, this week, right now, maybe, I might, I was thinking, for now, this month only, once, later, tomorrow.

Do not extract:
- one-day events
- current mood
- today's soreness
- today's weather
- meals eaten once
- skipped workouts
- temporary schedule disruptions
- vague future intentions
- curiosity questions
- assistant advice
- jokes or small talk
- facts about friends, family, influencers, or coaches
- unsupported guesses
- weak preferences stated casually without durability

Return only valid JSON in this exact schema:
{
  "memories": [
    {
      "memory": "Clear standalone long-term fact about the user.",
      "category": "fitness_goal | body_goal | workout_preference | workout_dislike | food_preference | dietary_restriction | nutrition_goal | schedule | injury_or_limitation | coaching_preference | tracking_preference | equipment | experience_level | lifestyle_constraint",
      "evidence": "Short quote or paraphrase from the user",
      "durability_signal": "Specific phrase that shows this is durable",
      "confidence": 0.0
    }
  ]
}

If there are no useful long-term semantic memories, return:
{"memories": []}

Each memory must be atomic. Store one fact per memory.

Few-shot examples:

Example 1

Conversation:
User: It was raining all day so I skipped my walk.
Coach: That's okay.
User: I usually walk every morning before work anyway.
Coach: Nice.
User: Yeah, morning workouts are easiest for me.

Output:
{
  "memories": [
    {
      "memory": "The user usually walks in the mornings before work.",
      "category": "schedule",
      "evidence": "I usually walk every morning before work",
      "durability_signal": "usually",
      "confidence": 0.95
    },
    {
      "memory": "The user prefers morning workouts.",
      "category": "workout_preference",
      "evidence": "morning workouts are easiest for me",
      "durability_signal": "easiest for me",
      "confidence": 0.9
    }
  ]
}

Example 2

Conversation:
User: I ate pizza and ice cream yesterday.
Coach: That's okay.
User: Normally I cook most of my meals at home.

Output:
{
  "memories": [
    {
      "memory": "The user usually cooks most meals at home.",
      "category": "food_preference",
      "evidence": "Normally I cook most of my meals at home",
      "durability_signal": "normally",
      "confidence": 0.94
    }
  ]
}

Example 3

Conversation:
User: I skipped the gym today because I felt tired.
User: I might start running next week.
User: It was really hot outside.

Output:
{
  "memories": []
}

Example 4

Conversation:
User: My friend Sarah follows keto.
Coach: Interesting.
User: I actually prefer eating rice and fruit.

Output:
{
  "memories": [
    {
      "memory": "The user prefers eating rice and fruit.",
      "category": "food_preference",
      "evidence": "I actually prefer eating rice and fruit",
      "durability_signal": "I actually prefer",
      "confidence": 0.93
    }
  ]
}

Example 5

Conversation:
User: I hate burpees.
Coach: Why?
User: Every workout plan that includes them makes me quit.
User: Squats are fine though.

Output:
{
  "memories": [
    {
      "memory": "The user dislikes burpees.",
      "category": "workout_dislike",
      "evidence": "I hate burpees",
      "durability_signal": "I hate",
      "confidence": 0.96
    },
    {
      "memory": "The user is comfortable performing squats.",
      "category": "workout_preference",
      "evidence": "Squats are fine though",
      "durability_signal": "are fine",
      "confidence": 0.86
    }
  ]
}

Example 6

Conversation:
User: I don't respond well to harsh feedback.
Coach: Understood.
User: Encouragement works much better for me.

Output:
{
  "memories": [
    {
      "memory": "The user prefers encouraging coaching feedback.",
      "category": "coaching_preference",
      "evidence": "Encouragement works much better for me",
      "durability_signal": "works much better for me",
      "confidence": 0.95
    }
  ]
}

Example 7

Conversation:
User: I used to run marathons.
Coach: Nice.
User: These days I mostly strength train and rarely run.

Output:
{
  "memories": [
    {
      "memory": "The user currently focuses mostly on strength training.",
      "category": "workout_preference",
      "evidence": "These days I mostly strength train",
      "durability_signal": "these days mostly",
      "confidence": 0.88
    },
    {
      "memory": "The user rarely runs currently.",
      "category": "workout_preference",
      "evidence": "rarely run",
      "durability_signal": "rarely",
      "confidence": 0.86
    }
  ]
}

Example 8

Conversation:
User: My shoulder is sore today from moving furniture.
Coach: Okay.
User: I do have a long-term knee issue that limits jumping exercises.

Output:
{
  "memories": [
    {
      "memory": "The user has a long-term knee issue that limits jumping exercises.",
      "category": "injury_or_limitation",
      "evidence": "I do have a long-term knee issue that limits jumping exercises",
      "durability_signal": "long-term",
      "confidence": 0.97
    }
  ]
}

Example 9

Conversation:
User: I work out at home.
Coach: What equipment do you have?
User: Adjustable dumbbells, resistance bands, and a bench.

Output:
{
  "memories": [
    {
      "memory": "The user works out at home.",
      "category": "lifestyle_constraint",
      "evidence": "I work out at home",
      "durability_signal": "work out at home",
      "confidence": 0.91
    },
    {
      "memory": "The user has adjustable dumbbells.",
      "category": "equipment",
      "evidence": "Adjustable dumbbells",
      "durability_signal": "equipment owned",
      "confidence": 0.94
    },
    {
      "memory": "The user has resistance bands.",
      "category": "equipment",
      "evidence": "resistance bands",
      "durability_signal": "equipment owned",
      "confidence": 0.94
    },
    {
      "memory": "The user has a bench.",
      "category": "equipment",
      "evidence": "a bench",
      "durability_signal": "equipment owned",
      "confidence": 0.94
    }
  ]
}

Example 10

Conversation:
User: Work has been crazy lately.
Coach: Sorry to hear that.
User: Yeah. I missed two workouts this week.
Coach: Happens.
User: Usually I train four mornings per week.
Coach: Good consistency.
User: I also prefer shorter workouts around 30 minutes.

Output:
{
  "memories": [
    {
      "memory": "The user typically trains four mornings per week.",
      "category": "schedule",
      "evidence": "Usually I train four mornings per week",
      "durability_signal": "usually",
      "confidence": 0.95
    },
    {
      "memory": "The user prefers workouts around 30 minutes.",
      "category": "workout_preference",
      "evidence": "I also prefer shorter workouts around 30 minutes",
      "durability_signal": "I also prefer",
      "confidence": 0.93
    }
  ]
}

Example 11

Conversation:
User: Hello.
Coach: Hi.
User: Tell me a joke.
Coach: Why did the dumbbell cross the road?
User: Haha.

Output:
{
  "memories": []
}

Example 12

Conversation:
User: I don't count calories because it makes me obsessive.
Coach: We can use a habit-based approach.
User: Yeah, portion guidance works better for me.

Output:
{
  "memories": [
    {
      "memory": "The user does not want to count calories.",
      "category": "tracking_preference",
      "evidence": "I don't count calories",
      "durability_signal": "I don't",
      "confidence": 0.95
    },
    {
      "memory": "Calorie counting makes the user feel obsessive.",
      "category": "tracking_preference",
      "evidence": "because it makes me obsessive",
      "durability_signal": "makes me obsessive",
      "confidence": 0.91
    },
    {
      "memory": "The user prefers portion guidance.",
      "category": "tracking_preference",
      "evidence": "portion guidance works better for me",
      "durability_signal": "works better for me",
      "confidence": 0.94
    }
  ]
}

Now extract memories from the provided conversation.
"""


## Data classes

These classes keep the evaluation code readable and make each object explicit.


In [ ]:
@dataclass
class ExpectedMemory:
    """Represents one human-labeled memory for a test case."""

    memory: str
    category: str
    evidence: str


@dataclass
class PromptTestCase:
    """Represents one conversation used to evaluate the extractor prompt."""

    test_id: str
    conversation_type: str
    conversation: str
    expected_memories: List[ExpectedMemory]
    trap: str


@dataclass
class MemoryEvaluationResult:
    """Represents scoring results for one evaluated test case."""

    test_id: str
    conversation_type: str
    true_positive_count: int
    false_positive_count: int
    false_negative_count: int
    precision: float
    recall: float
    f1: float
    category_accuracy: Optional[float]
    missed_expected: List[str]
    extra_predicted: List[str]
    trap: str


## Utility functions

These functions handle JSON parsing, model loading, text normalization, scoring, and output saving.


In [ ]:
def load_model_and_tokenizer(model_name: str) -> Tuple[Any, Any]:
    """Loads a Hugging Face causal language model and tokenizer."""

    loaded_tokenizer = AutoTokenizer.from_pretrained(model_name)
    loaded_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype="auto",
        device_map="auto",
    )
    loaded_model.eval()
    return loaded_model, loaded_tokenizer


def extract_json_object(raw_text: str) -> Dict[str, Any]:
    """Extracts the first valid JSON object from a model response."""

    cleaned_text = raw_text.strip()
    cleaned_text = re.sub(r"^```(?:json)?", "", cleaned_text).strip()
    cleaned_text = re.sub(r"```$", "", cleaned_text).strip()

    try:
        return json.loads(cleaned_text)
    except json.JSONDecodeError:
        json_match = re.search(r"\{.*\}", cleaned_text, flags=re.S)
        if not json_match:
            raise
        return json.loads(json_match.group(0))


def normalize_text(value: str) -> str:
    """Normalizes text for lightweight memory matching."""

    lowered_text = value.lower()
    alphanumeric_text = re.sub(r"[^a-z0-9\s]", " ", lowered_text)
    return re.sub(r"\s+", " ", alphanumeric_text).strip()


def calculate_token_f1(expected_text: str, predicted_text: str) -> float:
    """Calculates token-level F1 overlap between two memory strings."""

    expected_tokens = set(normalize_text(expected_text).split())
    predicted_tokens = set(normalize_text(predicted_text).split())

    if not expected_tokens and not predicted_tokens:
        return 1.0

    if not expected_tokens or not predicted_tokens:
        return 0.0

    common_token_count = len(expected_tokens.intersection(predicted_tokens))
    precision = common_token_count / len(predicted_tokens)
    recall = common_token_count / len(expected_tokens)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def safe_get_memories(model_output: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Returns model memories as a list, even when the model output is malformed."""

    memories = model_output.get("memories", [])

    if not isinstance(memories, list):
        return []

    valid_memories = []

    for memory in memories:
        if isinstance(memory, dict) and isinstance(memory.get("memory", ""), str):
            valid_memories.append(memory)

    return valid_memories


## Semantic memory extractor

The extractor class wraps the Hugging Face generation call and returns parsed JSON.

Generation uses temperature `0.0` by default because this is an evaluation task, not a creative writing task.


In [ ]:
class SemanticMemoryExtractor:
    """Runs a Qwen instruct model with the semantic memory extraction prompt."""

    def __init__(
        self,
        model_name: str,
        system_prompt: str,
        max_new_tokens: int = 768,
        temperature: float = 0.0,
    ) -> None:
        """Initializes model, tokenizer, and generation settings."""

        self.model_name = model_name
        self.system_prompt = system_prompt
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        self.model, self.tokenizer = load_model_and_tokenizer(model_name)

    def build_chat_prompt(self, conversation: str) -> str:
        """Builds the chat-formatted prompt expected by the Qwen instruct model."""

        messages = [
            {"role": "system", "content": self.system_prompt},
            {
                "role": "user",
                "content": f"Conversation:\n{conversation}\n\nReturn the semantic memory JSON now.",
            },
        ]

        return self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    def extract_memories(self, conversation: str) -> Dict[str, Any]:
        """Generates and parses semantic memories for one conversation."""

        chat_prompt = self.build_chat_prompt(conversation)
        model_inputs = self.tokenizer([chat_prompt], return_tensors="pt").to(self.model.device)

        generation_kwargs = {
            "max_new_tokens": self.max_new_tokens,
            "pad_token_id": self.tokenizer.eos_token_id,
        }

        if self.temperature > 0:
            generation_kwargs["do_sample"] = True
            generation_kwargs["temperature"] = self.temperature
        else:
            generation_kwargs["do_sample"] = False

        with torch.no_grad():
            generated_ids = self.model.generate(
                **model_inputs,
                **generation_kwargs,
            )

        response_ids = generated_ids[:, model_inputs.input_ids.shape[-1]:]
        raw_output = self.tokenizer.batch_decode(response_ids, skip_special_tokens=True)[0]

        try:
            parsed_output = extract_json_object(raw_output)
            parsed_output["raw_output"] = raw_output
            parsed_output["json_parse_error"] = None
            return parsed_output
        except Exception as error:
            return {
                "memories": [],
                "raw_output": raw_output,
                "json_parse_error": str(error),
            }


## Prompt Test Pack

The test data intentionally includes many negative cases.

This is important because the goal is not to extract the most memories. The goal is to avoid junk long-term memories.


In [ ]:
TEST_CASE_DATA = [
    {
        "test_id": "T001",
        "conversation_type": "strong_long_term",
        "conversation": "User: I’m trying to build strength without getting bulky. I usually lift dumbbells at home three times a week.\\nCoach: Great. What equipment do you have?\\nUser: Just adjustable dumbbells and a yoga mat.",
        "expected_memories": [
            {
                "memory": "The user wants to build strength without getting bulky.",
                "category": "fitness_goal",
                "evidence": "I’m trying to build strength without getting bulky"
            },
            {
                "memory": "The user usually lifts dumbbells at home three times per week.",
                "category": "schedule",
                "evidence": "I usually lift dumbbells at home three times a week"
            },
            {
                "memory": "The user has adjustable dumbbells.",
                "category": "equipment",
                "evidence": "adjustable dumbbells"
            },
            {
                "memory": "The user has a yoga mat.",
                "category": "equipment",
                "evidence": "a yoga mat"
            }
        ],
        "trap": "None"
    },
    {
        "test_id": "T002",
        "conversation_type": "temporary_only",
        "conversation": "User: It’s raining so hard today. I was going to run but maybe I’ll just watch Netflix.\\nCoach: No worries, you can do an indoor walk.\\nUser: Yeah maybe later.",
        "expected_memories": [],
        "trap": "Weather and temporary plan should be ignored"
    },
    {
        "test_id": "T003",
        "conversation_type": "strong_long_term",
        "conversation": "User: I hate burpees. Every program that includes them makes me quit.\\nCoach: Got it, we can avoid them.\\nUser: I’m okay with squats and lunges though.",
        "expected_memories": [
            {
                "memory": "The user dislikes burpees.",
                "category": "workout_dislike",
                "evidence": "I hate burpees"
            },
            {
                "memory": "The user is comfortable with squats.",
                "category": "workout_preference",
                "evidence": "I’m okay with squats"
            },
            {
                "memory": "The user is comfortable with lunges.",
                "category": "workout_preference",
                "evidence": "I’m okay with squats and lunges"
            }
        ],
        "trap": "None"
    },
    {
        "test_id": "T004",
        "conversation_type": "food_nutrition",
        "conversation": "User: I’m vegetarian, but I do eat eggs and dairy. I’m trying to get more protein because I feel hungry after workouts.",
        "expected_memories": [
            {
                "memory": "The user follows a vegetarian diet that includes eggs and dairy.",
                "category": "dietary_restriction",
                "evidence": "I’m vegetarian, but I do eat eggs and dairy"
            },
            {
                "memory": "The user wants to increase protein intake.",
                "category": "nutrition_goal",
                "evidence": "I’m trying to get more protein"
            }
        ],
        "trap": "Do not over-infer exact protein target"
    },
    {
        "test_id": "T005",
        "conversation_type": "injury_or_limitation",
        "conversation": "User: My knee has bothered me since an old basketball injury, so jumping exercises are risky for me.\\nCoach: We can use low-impact options.",
        "expected_memories": [
            {
                "memory": "The user has a knee limitation from an old basketball injury.",
                "category": "injury_or_limitation",
                "evidence": "My knee has bothered me since an old basketball injury"
            },
            {
                "memory": "Jumping exercises are risky for the user.",
                "category": "injury_or_limitation",
                "evidence": "jumping exercises are risky for me"
            }
        ],
        "trap": "None"
    },
    {
        "test_id": "T006",
        "conversation_type": "irrelevant_chitchat",
        "conversation": "User: Did you see the game last night? Wild finish.\\nCoach: I missed it.\\nUser: Anyway what should I eat before leg day?",
        "expected_memories": [],
        "trap": "Sports talk and a generic question should not be stored"
    },
    {
        "test_id": "T007",
        "conversation_type": "strong_long_term",
        "conversation": "User: I can work out only in the mornings before my kids wake up, usually around 6 am.\\nCoach: How long do you have?\\nUser: About 30 minutes max.",
        "expected_memories": [
            {
                "memory": "The user can only work out in the mornings before their kids wake up.",
                "category": "schedule",
                "evidence": "I can work out only in the mornings before my kids wake up"
            },
            {
                "memory": "The user usually works out around 6 am.",
                "category": "schedule",
                "evidence": "usually around 6 am"
            },
            {
                "memory": "The user usually has a maximum of 30 minutes for workouts.",
                "category": "schedule",
                "evidence": "About 30 minutes max"
            },
            {
                "memory": "The user's workout schedule is constrained by childcare responsibilities.",
                "category": "lifestyle_constraint",
                "evidence": "before my kids wake up"
            }
        ],
        "trap": "None"
    },
    {
        "test_id": "T008",
        "conversation_type": "coaching_preference",
        "conversation": "User: I’m new to the gym and machines make me nervous. I prefer simple instructions, not too much science at once.",
        "expected_memories": [
            {
                "memory": "The user is new to the gym.",
                "category": "experience_level",
                "evidence": "I’m new to the gym"
            },
            {
                "memory": "Gym machines make the user nervous.",
                "category": "experience_level",
                "evidence": "machines make me nervous"
            },
            {
                "memory": "The user prefers simple coaching instructions.",
                "category": "coaching_preference",
                "evidence": "I prefer simple instructions"
            },
            {
                "memory": "The user does not want too much science explained at once.",
                "category": "coaching_preference",
                "evidence": "not too much science at once"
            }
        ],
        "trap": "None"
    },
    {
        "test_id": "T009",
        "conversation_type": "temporary_only",
        "conversation": "User: I had pizza yesterday and felt guilty.\\nCoach: One meal is not a problem.\\nUser: True, I don’t usually eat pizza.",
        "expected_memories": [],
        "trap": "One-off meal and guilt should not be stored"
    },
    {
        "test_id": "T010",
        "conversation_type": "food_nutrition",
        "conversation": "User: I’m lactose intolerant, so whey shakes usually upset my stomach. Plant protein works better for me.",
        "expected_memories": [
            {
                "memory": "The user is lactose intolerant.",
                "category": "dietary_restriction",
                "evidence": "I’m lactose intolerant"
            },
            {
                "memory": "Whey protein shakes usually upset the user's stomach.",
                "category": "dietary_restriction",
                "evidence": "whey shakes usually upset my stomach"
            },
            {
                "memory": "Plant protein works better for the user.",
                "category": "food_preference",
                "evidence": "Plant protein works better for me"
            }
        ],
        "trap": "None"
    },
    {
        "test_id": "T011",
        "conversation_type": "strong_long_term",
        "conversation": "User: My goal is to run a 10K by the end of the year. Right now I can comfortably run 3K.",
        "expected_memories": [
            {
                "memory": "The user wants to run a 10K by the end of the year.",
                "category": "fitness_goal",
                "evidence": "My goal is to run a 10K by the end of the year"
            },
            {
                "memory": "The user can currently run about 3K comfortably.",
                "category": "experience_level",
                "evidence": "Right now I can comfortably run 3K"
            }
        ],
        "trap": "Current ability is useful baseline, not junk"
    },
    {
        "test_id": "T012",
        "conversation_type": "irrelevant_chitchat",
        "conversation": "User: I’m bored lol. Tell me a joke.\\nCoach: Why did the dumbbell cross the road?\\nUser: haha okay that was bad.",
        "expected_memories": [],
        "trap": "Jokes and boredom should be ignored"
    },
    {
        "test_id": "T013",
        "conversation_type": "lifestyle_constraint",
        "conversation": "User: I travel for work Monday to Thursday, so hotel workouts are easiest. I rarely have access to a full gym.",
        "expected_memories": [
            {
                "memory": "The user travels for work Monday to Thursday.",
                "category": "lifestyle_constraint",
                "evidence": "I travel for work Monday to Thursday"
            },
            {
                "memory": "Hotel workouts are easiest for the user.",
                "category": "workout_preference",
                "evidence": "hotel workouts are easiest"
            },
            {
                "memory": "The user rarely has access to a full gym while traveling.",
                "category": "equipment",
                "evidence": "I rarely have access to a full gym"
            }
        ],
        "trap": "None"
    },
    {
        "test_id": "T014",
        "conversation_type": "tracking_preference",
        "conversation": "User: I don’t count calories. It makes me obsessive. I prefer portion guidance or habit goals.",
        "expected_memories": [
            {
                "memory": "The user does not want to count calories.",
                "category": "tracking_preference",
                "evidence": "I don’t count calories"
            },
            {
                "memory": "Calorie counting makes the user feel obsessive.",
                "category": "tracking_preference",
                "evidence": "It makes me obsessive"
            },
            {
                "memory": "The user prefers portion guidance.",
                "category": "tracking_preference",
                "evidence": "I prefer portion guidance"
            },
            {
                "memory": "The user prefers habit goals.",
                "category": "tracking_preference",
                "evidence": "or habit goals"
            }
        ],
        "trap": "None"
    },
    {
        "test_id": "T015",
        "conversation_type": "temporary_only",
        "conversation": "User: Today my shoulders feel sore from painting the house.\\nCoach: Let’s do legs.\\nUser: Good idea.",
        "expected_memories": [],
        "trap": "Temporary soreness and assistant suggestion should be ignored"
    },
    {
        "test_id": "T016",
        "conversation_type": "strong_long_term",
        "conversation": "User: I’m training for hiking season. Uphill endurance and stronger glutes would help me most.",
        "expected_memories": [
            {
                "memory": "The user is training for hiking season.",
                "category": "fitness_goal",
                "evidence": "I’m training for hiking season"
            },
            {
                "memory": "The user wants better uphill endurance.",
                "category": "fitness_goal",
                "evidence": "Uphill endurance"
            },
            {
                "memory": "The user wants stronger glutes.",
                "category": "body_goal",
                "evidence": "stronger glutes would help me most"
            }
        ],
        "trap": "None"
    },
    {
        "test_id": "T017",
        "conversation_type": "food_nutrition",
        "conversation": "User: I fast until noon most days. I usually have black coffee in the morning and my first meal at lunch.",
        "expected_memories": [
            {
                "memory": "The user usually fasts until noon.",
                "category": "dietary_restriction",
                "evidence": "I fast until noon most days"
            },
            {
                "memory": "The user usually drinks black coffee in the morning.",
                "category": "food_preference",
                "evidence": "I usually have black coffee in the morning"
            },
            {
                "memory": "The user's first meal is usually lunch.",
                "category": "schedule",
                "evidence": "my first meal at lunch"
            }
        ],
        "trap": "None"
    },
    {
        "test_id": "T018",
        "conversation_type": "coaching_preference",
        "conversation": "User: Can you make the plan more encouraging? I shut down when feedback sounds harsh.",
        "expected_memories": [
            {
                "memory": "The user prefers encouraging coaching feedback.",
                "category": "coaching_preference",
                "evidence": "make the plan more encouraging"
            },
            {
                "memory": "The user responds poorly to harsh-sounding feedback.",
                "category": "coaching_preference",
                "evidence": "I shut down when feedback sounds harsh"
            }
        ],
        "trap": "None"
    },
    {
        "test_id": "T019",
        "conversation_type": "food_nutrition",
        "conversation": "User: I’m allergic to peanuts. Also I don’t really like almonds, but cashews are fine.",
        "expected_memories": [
            {
                "memory": "The user is allergic to peanuts.",
                "category": "dietary_restriction",
                "evidence": "I’m allergic to peanuts"
            },
            {
                "memory": "The user dislikes almonds.",
                "category": "food_preference",
                "evidence": "I don’t really like almonds"
            },
            {
                "memory": "The user is okay with cashews.",
                "category": "food_preference",
                "evidence": "cashews are fine"
            }
        ],
        "trap": "None"
    },
    {
        "test_id": "T020",
        "conversation_type": "temporary_only",
        "conversation": "User: What’s the weather in Mumbai tomorrow?\\nCoach: Looks humid.\\nUser: Ugh, okay.",
        "expected_memories": [],
        "trap": "Weather query should not be stored"
    },
    {
        "test_id": "T021",
        "conversation_type": "history_or_contradiction",
        "conversation": "User: I used to do CrossFit for two years, but now I want lower-impact strength training.",
        "expected_memories": [
            {
                "memory": "The user has about two years of CrossFit experience.",
                "category": "experience_level",
                "evidence": "I used to do CrossFit for two years"
            },
            {
                "memory": "The user now wants lower-impact strength training.",
                "category": "workout_preference",
                "evidence": "now I want lower-impact strength training"
            }
        ],
        "trap": "Do not treat CrossFit as current routine"
    },
    {
        "test_id": "T022",
        "conversation_type": "mixed_useful_and_junk",
        "conversation": "User: I only have 15 minutes today, but normally I like longer Sunday workouts.\\nCoach: How long on Sundays?\\nUser: Around 60 to 75 minutes.",
        "expected_memories": [
            {
                "memory": "The user normally likes longer Sunday workouts.",
                "category": "schedule",
                "evidence": "normally I like longer Sunday workouts"
            },
            {
                "memory": "The user's Sunday workouts are usually around 60 to 75 minutes.",
                "category": "schedule",
                "evidence": "Around 60 to 75 minutes"
            }
        ],
        "trap": "Ignore today-only 15 minute constraint"
    },
    {
        "test_id": "T023",
        "conversation_type": "food_nutrition",
        "conversation": "User: I’m trying to gain weight. I get full quickly, so smoothies are easier than huge meals.",
        "expected_memories": [
            {
                "memory": "The user is trying to gain weight.",
                "category": "body_goal",
                "evidence": "I’m trying to gain weight"
            },
            {
                "memory": "The user gets full quickly.",
                "category": "nutrition_goal",
                "evidence": "I get full quickly"
            },
            {
                "memory": "Smoothies are easier for the user than large meals.",
                "category": "food_preference",
                "evidence": "smoothies are easier than huge meals"
            }
        ],
        "trap": "None"
    },
    {
        "test_id": "T024",
        "conversation_type": "lifestyle_constraint",
        "conversation": "User: My apartment neighbors complain when I jump, so I need quiet workouts.\\nCoach: Low-impact it is.",
        "expected_memories": [
            {
                "memory": "The user needs quiet workouts because of apartment neighbors.",
                "category": "lifestyle_constraint",
                "evidence": "My apartment neighbors complain when I jump, so I need quiet workouts"
            },
            {
                "memory": "The user should avoid jumping or noisy exercises at home.",
                "category": "workout_preference",
                "evidence": "neighbors complain when I jump"
            }
        ],
        "trap": "None"
    },
    {
        "test_id": "T025",
        "conversation_type": "food_nutrition",
        "conversation": "User: I hate cooking. If meal prep takes more than 20 minutes, I won’t do it.",
        "expected_memories": [
            {
                "memory": "The user dislikes cooking.",
                "category": "food_preference",
                "evidence": "I hate cooking"
            },
            {
                "memory": "The user is unlikely to do meal prep that takes more than 20 minutes.",
                "category": "lifestyle_constraint",
                "evidence": "If meal prep takes more than 20 minutes, I won’t do it"
            }
        ],
        "trap": "None"
    },
    {
        "test_id": "T026",
        "conversation_type": "mixed_useful_and_junk",
        "conversation": "User: I’m Ramadan fasting this month, so my workout timing is weird.\\nCoach: We can adapt.\\nUser: Usually outside Ramadan I train after work.",
        "expected_memories": [
            {
                "memory": "Outside Ramadan, the user usually trains after work.",
                "category": "schedule",
                "evidence": "Usually outside Ramadan I train after work"
            }
        ],
        "trap": "Do not store Ramadan timing as permanent"
    },
    {
        "test_id": "T027",
        "conversation_type": "strong_long_term",
        "conversation": "User: I want bigger arms and shoulders. Legs are not my priority right now, but I know I should not skip them completely.",
        "expected_memories": [
            {
                "memory": "The user wants bigger arms.",
                "category": "body_goal",
                "evidence": "I want bigger arms"
            },
            {
                "memory": "The user wants bigger shoulders.",
                "category": "body_goal",
                "evidence": "and shoulders"
            },
            {
                "memory": "Leg development is not the user's current priority.",
                "category": "body_goal",
                "evidence": "Legs are not my priority right now"
            },
            {
                "memory": "The user does not want to skip leg training completely.",
                "category": "workout_preference",
                "evidence": "I should not skip them completely"
            }
        ],
        "trap": "Current priority is still useful coaching context"
    },
    {
        "test_id": "T028",
        "conversation_type": "injury_or_limitation",
        "conversation": "User: My doctor told me to avoid high-impact exercise because of my back. I’m comfortable with swimming and cycling.",
        "expected_memories": [
            {
                "memory": "The user's doctor told them to avoid high-impact exercise because of their back.",
                "category": "injury_or_limitation",
                "evidence": "My doctor told me to avoid high-impact exercise because of my back"
            },
            {
                "memory": "The user is comfortable with swimming.",
                "category": "workout_preference",
                "evidence": "I’m comfortable with swimming"
            },
            {
                "memory": "The user is comfortable with cycling.",
                "category": "workout_preference",
                "evidence": "and cycling"
            }
        ],
        "trap": "None"
    },
    {
        "test_id": "T029",
        "conversation_type": "coaching_preference",
        "conversation": "User: I like data. Show me reps, sets, RPE, and progress charts. Vague advice annoys me.",
        "expected_memories": [
            {
                "memory": "The user likes data-driven coaching.",
                "category": "coaching_preference",
                "evidence": "I like data"
            },
            {
                "memory": "The user wants reps, sets, RPE, and progress charts included in coaching.",
                "category": "tracking_preference",
                "evidence": "Show me reps, sets, RPE, and progress charts"
            },
            {
                "memory": "The user dislikes vague advice.",
                "category": "coaching_preference",
                "evidence": "Vague advice annoys me"
            }
        ],
        "trap": "None"
    },
    {
        "test_id": "T030",
        "conversation_type": "history_or_contradiction",
        "conversation": "User: Yesterday I met a coach named Ravi who said keto is best.\\nCoach: Do you want keto?\\nUser: Not really. I prefer rice and dal and don’t want to give up carbs.",
        "expected_memories": [
            {
                "memory": "The user prefers rice and dal.",
                "category": "food_preference",
                "evidence": "I prefer rice and dal"
            },
            {
                "memory": "The user does not want to give up carbohydrates.",
                "category": "food_preference",
                "evidence": "don’t want to give up carbs"
            },
            {
                "memory": "The user is not interested in keto.",
                "category": "food_preference",
                "evidence": "Not really"
            }
        ],
        "trap": "Ignore third-party coach opinion"
    }
]

In [ ]:
def build_prompt_test_cases(test_case_data: List[Dict[str, Any]]) -> List[PromptTestCase]:
    """Converts raw dictionaries into typed prompt test cases."""

    test_cases = []

    for item in test_case_data:
        expected_memories = [
            ExpectedMemory(
                memory=memory_item["memory"],
                category=memory_item["category"],
                evidence=memory_item["evidence"],
            )
            for memory_item in item["expected_memories"]
        ]

        test_cases.append(
            PromptTestCase(
                test_id=item["test_id"],
                conversation_type=item["conversation_type"],
                conversation=item["conversation"],
                expected_memories=expected_memories,
                trap=item["trap"],
            )
        )

    return test_cases


prompt_test_cases = build_prompt_test_cases(TEST_CASE_DATA)

print(f"Loaded {len(prompt_test_cases)} test cases.")


## Load the model

This cell downloads the Qwen model from Hugging Face and prepares the extractor.

If you change `MODEL_NAME`, rerun this cell.


In [ ]:
extractor = SemanticMemoryExtractor(
    model_name=MODEL_NAME,
    system_prompt=SYSTEM_PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=GENERATION_TEMPERATURE,
)

print(f"Loaded extractor with model: {MODEL_NAME}")


## Run extraction on all test cases

This step may take a while depending on the model size and available GPU.


In [ ]:
def run_test_cases(
    memory_extractor: SemanticMemoryExtractor,
    test_cases: List[PromptTestCase],
) -> List[Dict[str, Any]]:
    """Runs semantic memory extraction for all prompt test cases."""

    results = []

    for test_case in tqdm(test_cases):
        model_output = memory_extractor.extract_memories(test_case.conversation)
        predicted_memories = safe_get_memories(model_output)

        results.append(
            {
                "test_id": test_case.test_id,
                "conversation_type": test_case.conversation_type,
                "conversation": test_case.conversation,
                "expected_memories": [memory.__dict__ for memory in test_case.expected_memories],
                "predicted_memories": predicted_memories,
                "raw_output": model_output.get("raw_output", ""),
                "json_parse_error": model_output.get("json_parse_error"),
                "trap": test_case.trap,
            }
        )

    return results


extraction_results = run_test_cases(extractor, prompt_test_cases)

results_preview = pd.DataFrame(
    [
        {
            "test_id": result["test_id"],
            "conversation_type": result["conversation_type"],
            "expected_count": len(result["expected_memories"]),
            "predicted_count": len(result["predicted_memories"]),
            "json_parse_error": result["json_parse_error"],
        }
        for result in extraction_results
    ]
)

display(results_preview)


## Evaluation functions

The matching is intentionally lightweight for prompt iteration.

It compares expected and predicted memory strings using token-overlap F1. This is not a perfect semantic judge, but it helps identify false positives, false negatives, and prompt regressions quickly.

Manual review is still required for borderline cases.


In [ ]:
def match_expected_and_predicted_memories(
    expected_memories: List[Dict[str, Any]],
    predicted_memories: List[Dict[str, Any]],
    match_threshold: float,
) -> Tuple[List[Tuple[int, int, float]], List[int], List[int]]:
    """Matches predicted memories to expected memories using token-level F1."""

    matched_expected_indexes = set()
    matched_predicted_indexes = set()
    matches = []

    for expected_index, expected_memory in enumerate(expected_memories):
        best_predicted_index = None
        best_score = 0.0

        for predicted_index, predicted_memory in enumerate(predicted_memories):
            if predicted_index in matched_predicted_indexes:
                continue

            score = calculate_token_f1(
                expected_memory.get("memory", ""),
                predicted_memory.get("memory", ""),
            )

            if score > best_score:
                best_score = score
                best_predicted_index = predicted_index

        if best_predicted_index is not None and best_score >= match_threshold:
            matched_expected_indexes.add(expected_index)
            matched_predicted_indexes.add(best_predicted_index)
            matches.append((expected_index, best_predicted_index, best_score))

    missed_expected_indexes = [
        index for index in range(len(expected_memories))
        if index not in matched_expected_indexes
    ]

    extra_predicted_indexes = [
        index for index in range(len(predicted_memories))
        if index not in matched_predicted_indexes
    ]

    return matches, missed_expected_indexes, extra_predicted_indexes


def calculate_category_accuracy(
    expected_memories: List[Dict[str, Any]],
    predicted_memories: List[Dict[str, Any]],
    matches: List[Tuple[int, int, float]],
) -> Optional[float]:
    """Calculates category accuracy for matched memories only."""

    if not matches:
        return None

    correct_category_count = 0

    for expected_index, predicted_index, _ in matches:
        expected_category = expected_memories[expected_index].get("category", "")
        predicted_category = predicted_memories[predicted_index].get("category", "")

        if expected_category == predicted_category:
            correct_category_count += 1

    return correct_category_count / len(matches)


def evaluate_single_result(
    result: Dict[str, Any],
    match_threshold: float,
) -> MemoryEvaluationResult:
    """Evaluates one extraction result against expected memories."""

    expected_memories = result["expected_memories"]
    predicted_memories = result["predicted_memories"]

    matches, missed_indexes, extra_indexes = match_expected_and_predicted_memories(
        expected_memories=expected_memories,
        predicted_memories=predicted_memories,
        match_threshold=match_threshold,
    )

    true_positive_count = len(matches)
    false_positive_count = len(extra_indexes)
    false_negative_count = len(missed_indexes)

    precision = (
        true_positive_count / (true_positive_count + false_positive_count)
        if true_positive_count + false_positive_count > 0
        else 1.0
    )

    recall = (
        true_positive_count / (true_positive_count + false_negative_count)
        if true_positive_count + false_negative_count > 0
        else 1.0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    category_accuracy = calculate_category_accuracy(
        expected_memories=expected_memories,
        predicted_memories=predicted_memories,
        matches=matches,
    )

    missed_expected = [
        expected_memories[index].get("memory", "")
        for index in missed_indexes
    ]

    extra_predicted = [
        predicted_memories[index].get("memory", "")
        for index in extra_indexes
    ]

    return MemoryEvaluationResult(
        test_id=result["test_id"],
        conversation_type=result["conversation_type"],
        true_positive_count=true_positive_count,
        false_positive_count=false_positive_count,
        false_negative_count=false_negative_count,
        precision=precision,
        recall=recall,
        f1=f1,
        category_accuracy=category_accuracy,
        missed_expected=missed_expected,
        extra_predicted=extra_predicted,
        trap=result["trap"],
    )


def evaluate_all_results(
    results: List[Dict[str, Any]],
    match_threshold: float,
) -> List[MemoryEvaluationResult]:
    """Evaluates all extraction results."""

    return [
        evaluate_single_result(result, match_threshold)
        for result in results
    ]


## Summary metrics

For semantic memory extraction, the most important metric is usually false positives.

A model that stores too many weak memories will create junk memory data.


In [ ]:
evaluation_results = evaluate_all_results(
    results=extraction_results,
    match_threshold=MATCH_THRESHOLD,
)

evaluation_rows = [evaluation_result.__dict__ for evaluation_result in evaluation_results]
evaluation_dataframe = pd.DataFrame(evaluation_rows)

json_parse_rate = sum(
    result["json_parse_error"] is None for result in extraction_results
) / len(extraction_results)

summary_metrics = {
    "model_name": MODEL_NAME,
    "test_case_count": len(extraction_results),
    "json_parse_rate": json_parse_rate,
    "average_precision": evaluation_dataframe["precision"].mean(),
    "average_recall": evaluation_dataframe["recall"].mean(),
    "average_f1": evaluation_dataframe["f1"].mean(),
    "total_false_positives": int(evaluation_dataframe["false_positive_count"].sum()),
    "total_false_negatives": int(evaluation_dataframe["false_negative_count"].sum()),
    "average_category_accuracy": evaluation_dataframe["category_accuracy"].dropna().mean(),
}

summary_metrics_dataframe = pd.DataFrame([summary_metrics])
display(summary_metrics_dataframe)


## Error analysis table

Use this table to review exactly where the prompt failed.

Pay special attention to:

1. Temporary facts stored as long-term memory
2. Weather or chit-chat stored as memory
3. Assistant advice stored as user memory
4. Third-party facts stored as user memory
5. Too many weak memories extracted


In [ ]:
def create_error_analysis_table(
    extraction_results: List[Dict[str, Any]],
    evaluation_results: List[MemoryEvaluationResult],
) -> pd.DataFrame:
    """Creates a readable table for manual prompt error analysis."""

    rows = []

    for extraction_result, evaluation_result in zip(extraction_results, evaluation_results):
        rows.append(
            {
                "test_id": extraction_result["test_id"],
                "conversation_type": extraction_result["conversation_type"],
                "conversation": extraction_result["conversation"],
                "expected_memories": [
                    memory.get("memory", "")
                    for memory in extraction_result["expected_memories"]
                ],
                "predicted_memories": [
                    memory.get("memory", "")
                    for memory in extraction_result["predicted_memories"]
                ],
                "missed_expected": evaluation_result.missed_expected,
                "extra_predicted": evaluation_result.extra_predicted,
                "precision": round(evaluation_result.precision, 3),
                "recall": round(evaluation_result.recall, 3),
                "f1": round(evaluation_result.f1, 3),
                "category_accuracy": (
                    round(evaluation_result.category_accuracy, 3)
                    if evaluation_result.category_accuracy is not None
                    else None
                ),
                "json_parse_error": extraction_result["json_parse_error"],
                "trap": extraction_result["trap"],
            }
        )

    return pd.DataFrame(rows)


error_analysis_dataframe = create_error_analysis_table(
    extraction_results=extraction_results,
    evaluation_results=evaluation_results,
)

display(error_analysis_dataframe)


## False-positive review

This view focuses only on cases where the extractor created extra memories.

These are the most important failures for your use case.


In [ ]:
false_positive_review = error_analysis_dataframe[
    error_analysis_dataframe["extra_predicted"].apply(lambda value: len(value) > 0)
].copy()

display(false_positive_review[
    [
        "test_id",
        "conversation_type",
        "extra_predicted",
        "trap",
        "conversation",
    ]
])


## False-negative review

This view shows memories that the model missed.

Only reduce strictness if false positives are already low.


In [ ]:
false_negative_review = error_analysis_dataframe[
    error_analysis_dataframe["missed_expected"].apply(lambda value: len(value) > 0)
].copy()

display(false_negative_review[
    [
        "test_id",
        "conversation_type",
        "missed_expected",
        "conversation",
    ]
])


## Save evaluation outputs

This cell saves:

1. Predictions CSV
2. Error analysis CSV
3. Raw outputs JSONL

Use these files to compare different prompt versions or model choices.


In [ ]:
def save_evaluation_outputs(
    extraction_results: List[Dict[str, Any]],
    error_analysis_dataframe: pd.DataFrame,
    predictions_csv_path: Path,
    error_analysis_csv_path: Path,
    raw_outputs_jsonl_path: Path,
) -> None:
    """Saves predictions, error analysis, and raw model outputs."""

    prediction_rows = []

    for result in extraction_results:
        prediction_rows.append(
            {
                "test_id": result["test_id"],
                "conversation_type": result["conversation_type"],
                "conversation": result["conversation"],
                "expected_memories": json.dumps(result["expected_memories"], ensure_ascii=False),
                "predicted_memories": json.dumps(result["predicted_memories"], ensure_ascii=False),
                "json_parse_error": result["json_parse_error"],
                "trap": result["trap"],
            }
        )

    predictions_dataframe = pd.DataFrame(prediction_rows)
    predictions_dataframe.to_csv(predictions_csv_path, index=False)
    error_analysis_dataframe.to_csv(error_analysis_csv_path, index=False)

    with raw_outputs_jsonl_path.open("w", encoding="utf-8") as output_file:
        for result in extraction_results:
            output_file.write(json.dumps(result, ensure_ascii=False) + "\n")


save_evaluation_outputs(
    extraction_results=extraction_results,
    error_analysis_dataframe=error_analysis_dataframe,
    predictions_csv_path=PREDICTIONS_CSV_PATH,
    error_analysis_csv_path=ERROR_ANALYSIS_CSV_PATH,
    raw_outputs_jsonl_path=RAW_OUTPUTS_JSONL_PATH,
)

print(f"Saved predictions to: {PREDICTIONS_CSV_PATH}")
print(f"Saved error analysis to: {ERROR_ANALYSIS_CSV_PATH}")
print(f"Saved raw outputs to: {RAW_OUTPUTS_JSONL_PATH}")


## Prompt improvement guide

Use the evaluation results to improve the prompt.

### If false positives are high

Try:

```text
Add more negative few-shot examples.
Strengthen the temporary and one-off rejection rules.
Require stronger durability signals.
Reject weak phrases like maybe, today, yesterday, this week, or right now.
Add examples where the correct output is {"memories": []}.
```

### If false negatives are high

Try:

```text
Add positive few-shot examples for the missed category.
Make category definitions clearer.
Reduce strictness only after false positives are low.
Add examples with stable but subtle memories.
```

### If JSON parse errors are high

Try:

```text
Repeat the JSON-only instruction near the end of the prompt.
Use temperature 0.0.
Reduce the number of few-shot examples if the prompt is too long.
Keep the schema simple.
```

### If category accuracy is low

Try:

```text
Add category definitions.
Add examples with similar memories but different categories.
Remove overlapping categories if they cause confusion.
```


## Recommended next experiments

1. Run this notebook with `Qwen/Qwen2.5-1.5B-Instruct`.
2. Save the output files.
3. Change only the prompt and rerun.
4. Compare false-positive counts.
5. Try `Qwen/Qwen2.5-7B-Instruct` if GPU memory allows.
6. Add more negative cases if the model stores junk data.
7. Add domain-specific cases from your real coaching conversations.

The main target is:

```text
precision >= 0.90
recall >= 0.70
false positives as low as possible
```
